# Multipole-to-particle evaluation (M2P)

## Purpose

M2P evaluates a multipole source representation directly at a distant target.
It is especially useful for validating P2M and M2M independently of local
expansions. This notebook studies convergence with expansion order, target
distance, and the geometric ratio of source radius to target distance.

## Mathematical definition

For $R=x-c_s$, the truncated potential is

$$\phi_p(x)=\sum_{|\alpha|\le p}M_\alpha D_\alpha G(R),
\qquad H_p=-\nabla\phi_p.$$

The expansion converges when the target lies outside the source cluster;
smaller source-radius/target-distance ratios generally converge faster.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import cdfmm

try:
    from example_utils import (
        direct_fields,
        draw_box_3d,
        error_metrics,
        finish_3d_axes,
        local_fields,
        multipole_fields,
        new_3d_figure,
        nodes_at_level,
        plot_coefficients_by_degree,
        random_unit_vectors,
        relative_error,
        set_axes_equal,
        vec3_to_array,
    )
except ModuleNotFoundError:
    # This path is used when the kernel starts in the repository root.
    from examples.notebooks.example_utils import (
        direct_fields,
        draw_box_3d,
        error_metrics,
        finish_3d_axes,
        local_fields,
        multipole_fields,
        new_3d_figure,
        nodes_at_level,
        plot_coefficients_by_degree,
        random_unit_vectors,
        relative_error,
        set_axes_equal,
        vec3_to_array,
    )

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True})

## User parameters

In [ ]:
n_sources = 100
maximum_order = 6
source_box_half_width = 0.45
minimum_target_distance = 1.5
maximum_target_distance = 7.0
n_target_distances = 14
random_seed = 42
source_centre = np.zeros(3)

## Problem setup

In [ ]:
rng = np.random.default_rng(random_seed)
source_positions = rng.uniform(
    -source_box_half_width,
    source_box_half_width,
    size=(n_sources, 3),
)
dipole_moments = rng.normal(size=(n_sources, 3))
source_radius = np.max(np.linalg.norm(source_positions - source_centre, axis=1))

target_distances = np.linspace(
    minimum_target_distance,
    maximum_target_distance,
    n_target_distances,
)
target_direction = np.array([1.0, 0.35, -0.2])
target_direction /= np.linalg.norm(target_direction)
target_positions = target_distances[:, np.newaxis] * target_direction
reference_fields = direct_fields(target_positions, source_positions, dipole_moments)

print(f"Sources: {n_sources}")
print(f"Source radius: {source_radius:.3f}")
print(f"Nearest target distance: {target_distances[0]:.3f}")
print(f"Largest source-radius/distance ratio: {source_radius / target_distances[0]:.3f}")

## Evaluation and convergence diagnostics

In [ ]:
orders = np.arange(1, maximum_order + 1)
errors_by_order = []

for order in orders:
    multipole = cdfmm.p2m_dipole(
        source_centre,
        source_positions,
        dipole_moments,
        order=order,
    )
    approximate_fields = multipole_fields(
        target_positions,
        multipole,
        source_centre,
        order,
    )
    errors_by_order.append(relative_error(approximate_fields, reference_fields))

errors_by_order = np.asarray(errors_by_order)
print(" p   coefficients   mean relative error   maximum relative error")
for order, errors in zip(orders, errors_by_order):
    coefficient_count = len(cdfmm.multi_indices(order))
    print(f" {order:1d}   {coefficient_count:12d}   {np.mean(errors):19.6e}   {np.max(errors):22.6e}")

## Visualisation

In [ ]:
figure, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].semilogy(orders, np.sqrt(np.mean(errors_by_order**2, axis=1)), "o-")
axes[0].set_xlabel("Expansion order p")
axes[0].set_ylabel("RMS relative field error")
axes[0].set_title("Error versus expansion order")

for order_index in [0, 2, 4, maximum_order - 1]:
    axes[1].semilogy(
        target_distances,
        errors_by_order[order_index],
        marker="o",
        label=f"p={orders[order_index]}",
    )
axes[1].set_xlabel("Target distance")
axes[1].set_ylabel("Relative field error")
axes[1].set_title("Error versus distance")
axes[1].legend()

distance_ratios = source_radius / target_distances
for order_index in [0, 2, 4, maximum_order - 1]:
    axes[2].semilogy(
        distance_ratios,
        errors_by_order[order_index],
        marker="o",
        label=f"p={orders[order_index]}",
    )
axes[2].set_xlabel("Source radius / target distance")
axes[2].set_ylabel("Relative field error")
axes[2].set_title("Error versus geometric ratio")
axes[2].legend()

figure.tight_layout()

## Interactive experiment

In [ ]:
from ipywidgets import FloatSlider, IntSlider, interact


def inspect_m2p(order=4, target_distance=3.0):
    multipole = cdfmm.p2m_dipole(
        source_centre,
        source_positions,
        dipole_moments,
        order=order,
    )
    target = target_distance * target_direction
    approximate = cdfmm.m2p(
        multipole,
        source_centre,
        target,
        order=order,
        output="field",
    )["H"]
    reference = cdfmm.p2p_dipole_sum(
        target,
        source_positions,
        dipole_moments,
        output="field",
    )["H"]
    error = relative_error(approximate[np.newaxis, :], reference[np.newaxis, :])[0]
    print(f"coefficient count = {len(multipole)}")
    print(f"source-radius/distance = {source_radius / target_distance:.3f}")
    print(f"relative field error = {error:.6e}")


interact(
    inspect_m2p,
    order=IntSlider(min=1, max=8, step=1, value=4),
    target_distance=FloatSlider(min=1.0, max=8.0, step=0.25, value=3.0),
)

## What to observe

Increasing order reduces truncation error, and moving the target farther from
the source cluster improves convergence at fixed order. The ratio plot is the
most transferable diagnostic because it expresses geometric separation rather
than a particular choice of length unit.